# Reproducibility copy of original Colab notebook
This notebook preserves the authors' analytical workflow, with **saved outputs removed** and credentials/path references sanitized.
It has not been validated end-to-end in a clean environment. Some cells depend on legacy intermediate variables, remote IUCN access, or private screening exports. See `docs/reproducibility.md`.


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
url = ROOT / 'data' / 'processed' / 'study_extraction.csv'


In [ ]:
words_to_remove = [
    "Paper Information",
    "ML Pipeline Information",
    "Dataset Information",
    "Geographic & Taxonomic Information",
    "Performance Information",
    "Synthesis Information"
]

In [ ]:
df = pd.read_csv(url)

In [ ]:
def clean_column(col_name):
    for word in words_to_remove:
        if col_name.startswith(word) and col_name != "Data quality control":
            col_name = col_name.replace(word, "").strip()
    return col_name

In [ ]:
df.columns = [clean_column(col) for col in df.columns]

In [ ]:
# Convert numeric fields
numeric_cols = [
    "Input Segment Length Seconds", "Dataset Size Train",
    "Dataset Size Validation", "Dataset Size Testing",
    "Total Duration Hours", "Accuracy", "Precision",
    "Recall", "Specificity", "F1 Score", "UAR"
]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Paper Information

In [ ]:
df.head()

In [ ]:
df.columns

## Year

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, y="Year", order=df["Year"].value_counts().index, color="C0")
plt.title("Papers Published Over the Years")
plt.xlabel("Count")
plt.ylabel("Year")
plt.tight_layout()
plt.show()

# ML Pipeline Information

## ML Algortihm Category

In [ ]:
# Split models and flatten
model_series = df["ML Algorithm Category"].dropna().str.split(",").explode().str.strip()

plt.figure(figsize=(10, 6))
sns.countplot(y=model_series, order=model_series.value_counts().index, color="C0")
plt.title("Count of Specific ML Algorithms Used")
plt.xlabel("Count")
plt.ylabel("ML Algorithm Specific")
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart
data_counts = df["ML Algorithm Category"].dropna().str.split(",").explode().str.strip().value_counts()

plt.figure(figsize=(8, 8))
plt.pie(
    data_counts,
    labels=data_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    explode=[0.05] * len(data_counts)
)
plt.title("Data Representation Distribution")
plt.tight_layout()
plt.show()

In [ ]:
titles = df['Title']
algorithms = df['ML Algorithm Category']

# Create the node list
all_nodes = pd.concat([titles, algorithms]).dropna().unique().tolist()
node_indices = {node: idx for idx, node in enumerate(all_nodes)}

# Create source and target indices
source_indices = titles.map(node_indices)
target_indices = algorithms.map(node_indices)

# Filter out any missing links
valid = source_indices.notna() & target_indices.notna()

source = source_indices[valid].astype(int).tolist()
target = target_indices[valid].astype(int).tolist()
value = [1]*len(source)

# Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=15,
        line=dict(color="black", width=0.5),
        label=all_nodes,
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Papers → ML Algorithms Category", font_size=10)
fig.show()

## ML Algorithm Specific

In [ ]:
# Split models and flatten
model_series = df["ML Algorithm Specific"].dropna().str.split(";").explode().str.strip()

plt.figure(figsize=(10, 10))
sns.countplot(y=model_series, order=model_series.value_counts().index, color="C0")
plt.title("Count of Specific ML Algorithms Used")
plt.xlabel("Count")
plt.ylabel("ML Algorithm Specific")
plt.tight_layout()
plt.show()

In [ ]:
df["ML Algorithm Specific"].unique()

In [ ]:
algorithms = df["ML Algorithm Specific"].dropna().str.split(";").explode().str.strip()

group_mapping = {
    'ResNet50': 'ResNet',
    'ResNet50V2': 'ResNet',
    'ResNet18': 'ResNet',
    'ResNet101': 'ResNet',
    'ResNet101V2': 'ResNet',
    'ResNet152V2': 'ResNet',
    'ResNet22': 'ResNet',
    'WideResNet28-2': 'ResNet',

    'MobileNetV2': 'MobileNet',
    'MobileNetV1': 'MobileNet',

    'DenseNet121': 'DenseNet',
    'DenseNet169': 'DenseNet',
    'DenseNet201': 'DenseNet',

    'InceptionResNetV2': 'Inception',
    'InceptionV3': 'Inception',

    'VGG16': 'VGG',
    'VGGish': 'VGG',

    'CNN': 'CNN',
    'DCNN': 'CNN',

    'BirdNET': 'BirdNET',

    'SVM': 'Linear Model',
    'KNN': 'KNN',
    'LR': 'Linear Model',
    'LDA': 'Linear Model',
    'DT': 'Decision Tree',
    'XGB': 'Ensemble Learning',
    'RF': 'Ensemble Learning',

    'GNB': 'Naive Bayes',
    'MLP': 'MLP',
    'RNN': 'RNN',
    'LSTM': 'LSTM',
    'BiLSTM': 'BiLSTM',
}

In [ ]:
algorithms_grouped = algorithms.map(lambda x: group_mapping.get(x.strip(), x.strip()))
data_counts = algorithms_grouped.value_counts()

# Plot
plt.figure(figsize=(12, 12))
plt.pie(
    data_counts,
    labels=data_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    explode=[0.05] * len(data_counts)
)
plt.title("Top ML Algorithm Types (Grouped)")
plt.tight_layout()
plt.show()

In [ ]:
df_exploded = df.dropna(subset=["ML Algorithm Specific"]).copy()
df_exploded["ML Algorithm Specific"] = df_exploded["ML Algorithm Specific"].str.split(";")
df_exploded = df_exploded.explode("ML Algorithm Specific")
df_exploded["ML Algorithm Specific"] = df_exploded["ML Algorithm Specific"].str.strip()

titles = df_exploded['Title']
algorithms = df_exploded['ML Algorithm Specific']

# Create the node list
all_nodes = pd.concat([titles, algorithms]).dropna().unique().tolist()
node_indices = {node: idx for idx, node in enumerate(all_nodes)}

# Map titles and algorithms to node indices
source = titles.map(node_indices).astype(int).tolist()
target = algorithms.map(node_indices).astype(int).tolist()
value = [1] * len(source)

# Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=15,
        line=dict(color="black", width=0.5),
        label=all_nodes,
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Papers → ML Algorithms", font_size=10)
fig.show()

## Data Representation

In [ ]:
# Count plots
plt.figure(figsize=(10, 5))
sns.countplot(
    data=df,
    y="Data Representation",
    order=df["Data Representation"].value_counts().index,
    color="C0"
)
plt.title("Data Representation Distribution")
plt.xlabel("Count")
plt.ylabel("Data Representation")
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart
data_counts = df["Data Representation"].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(
    data_counts,
    labels=data_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    explode=[0.05] * len(data_counts)
)
plt.title("Data Representation Distribution")
plt.tight_layout()
plt.show()

In [ ]:
df_exploded = df.dropna(subset=["ML Algorithm Category", "Data Representation"]).copy()
df_exploded["ML Algorithm Category"] = df_exploded["ML Algorithm Category"].str.split(",")
df_exploded = df_exploded.explode("ML Algorithm Category")
df_exploded["ML Algorithm Category"] = df_exploded["ML Algorithm Category"].str.strip()

titles = df_exploded['Title']
algorithms = df_exploded['ML Algorithm Category']
data_rep = df_exploded['Data Representation']

# Create all nodes (Title + Algorithm + Data Representation)
all_nodes = pd.concat([titles, algorithms, data_rep]).dropna().unique().tolist()
node_indices = {node: idx for idx, node in enumerate(all_nodes)}

# Title → Algorithm
source1 = titles.map(node_indices).astype(int).tolist()
target1 = algorithms.map(node_indices).astype(int).tolist()

# Algorithm → Data Representation
source2 = algorithms.map(node_indices).astype(int).tolist()
target2 = data_rep.map(node_indices).astype(int).tolist()

# Combine all links
source = source1 + source2
target = target1 + target2
value = [1]*len(source)

# Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=15,
        line=dict(color="black", width=0.5),
        label=all_nodes,
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Papers → ML Algorithms Category → Data Representation", font_size=10)
fig.show()

## Preprocessing Methods

In [ ]:
model_series = df["Preprocessing Methods"].dropna().str.split(",").explode().str.strip()

plt.figure(figsize=(10, 6))
sns.countplot(y=model_series, order=model_series.value_counts().index, color="C0")
plt.title("Count of Preprocessing Methods Used")
plt.xlabel("Count")
plt.ylabel("Preprocessing Methods")
plt.tight_layout()
plt.show()

In [ ]:
df_exploded = df.dropna(subset=["ML Algorithm Category", "Preprocessing Methods"]).copy()
df_exploded["ML Algorithm Category"] = df_exploded["ML Algorithm Category"].str.split(",")
df_exploded = df_exploded.explode("ML Algorithm Category")
df_exploded["ML Algorithm Category"] = df_exploded["ML Algorithm Category"].str.strip()

titles = df_exploded['Title']
algorithms = df_exploded['ML Algorithm Category']
data_rep = df_exploded['Preprocessing Methods'].dropna().str.split(",").explode().str.strip()

# Create all nodes (Title + Algorithm + Preprocessing Methods)
all_nodes = pd.concat([titles, algorithms, data_rep]).dropna().unique().tolist()
node_indices = {node: idx for idx, node in enumerate(all_nodes)}

# Title → Algorithm
source1 = titles.map(node_indices).astype(int).tolist()
target1 = algorithms.map(node_indices).astype(int).tolist()

# Algorithm → Preprocessing Methods
source2 = algorithms.map(node_indices).astype(int).tolist()
target2 = data_rep.map(node_indices).astype(int).tolist()

# Combine all links
source = source1 + source2
target = target1 + target2
value = [1]*len(source)

# Build the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=15,
        line=dict(color="black", width=0.5),
        label=all_nodes,
    ),
    link=dict(
        source=source,
        target=target,
        value=value
    ))])

fig.update_layout(title_text="Papers → ML Algorithms Category → Preprocessing Methods", font_size=10)
fig.show()

# Dataset Information

## Input Length (Seconds)

In [ ]:
segment_lengths = df['Input Segment Length Seconds'].dropna()

sns.set_theme(style="whitegrid")

# Create the plot
plt.figure(figsize=(14, 6))
ax = sns.countplot(
    x=segment_lengths,
    order=segment_lengths.value_counts().sort_index().index,
    color="C0"
)

# Title and labels
plt.title('Distribution of Input Segment Lengths (Each Second)', fontsize=18)
plt.xlabel('Input Segment Length (seconds)')
plt.ylabel('Number of Papers')

# Rotate x labels
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)

plt.tight_layout()
plt.show()


## Dataset Size

In [ ]:
df["Dataset Size Train"]

In [ ]:
dataset_cols = ['Dataset Size Train', 'Dataset Size Validation', 'Dataset Size Testing']
df_melted = df[dataset_cols].melt(var_name="Dataset Split", value_name="Size").dropna()

sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.violinplot(x="Dataset Split", y="Size", data=df_melted, color="C0", inner="box")

plt.title('Dataset Size Distribution: Train, Validation, Test', fontsize=18)
plt.xlabel('Dataset Split', fontsize=14)
plt.ylabel('Number of Samples', fontsize=14)

plt.grid(True)
plt.tight_layout()
plt.show()

## Total duration hours

In [ ]:
segment_lengths = df['Total Duration Hours'].dropna()

sns.set_theme(style="whitegrid")

# Create the plot
plt.figure(figsize=(14, 6))
ax = sns.countplot(
    x=segment_lengths,
    order=segment_lengths.value_counts().sort_index().index,
    color="C0"
)

# Title and labels
plt.title('Distribution of Total Duration of the dataset', fontsize=18)
plt.xlabel('Total Duration of the dataset (Hours)')
plt.ylabel('Number of Papers')

# Rotate x labels
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)

plt.tight_layout()
plt.show()

## Data Availability

In [ ]:
df['Data Availability']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare data
availability_counts = df['Data Availability'].value_counts()

# Set style
sns.set_theme(style="whitegrid")

# Bar plot
plt.figure(figsize=(8, 6))
sns.barplot(x=availability_counts.index, y=availability_counts.values, color="C0")

plt.title('Data Availability in Papers', fontsize=18)
plt.xlabel('Data Availability Type', fontsize=14)
plt.ylabel('Number of Papers', fontsize=14)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart
data_counts = df["Data Availability"].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(
    data_counts,
    labels=data_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    explode=[0.05] * len(data_counts)
)
plt.title("Data Availability Distribution")
plt.tight_layout()
plt.show()

# Geographic & Taxonomic Information

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Clean species (split and standardize like before)
df_species = df.dropna(subset=["Species", "Data Availability"]).copy()
df_species["Species"] = df_species["Species"].str.split(";")
df_species = df_species.explode("Species")
df_species["Species"] = df_species["Species"].str.strip().str.title()

# Create a count table
species_availability = df_species.groupby(["Species", "Data Availability"]).size().unstack(fill_value=0)

# Select top N species
top_species = species_availability.sum(axis=1).nlargest(10).index
species_availability = species_availability.loc[top_species]

# Plot
species_availability.plot(kind="bar", stacked=False, figsize=(14, 8), colormap="viridis")

plt.title('Data Availability by Species', fontsize=18)
plt.xlabel('Species', fontsize=14)
plt.ylabel('Number of Papers', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.legend(title="Data Availability")
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
df.columns

## Taxa Order

In [ ]:
df_taxa = df['Taxa Order'].dropna().str.split(';').explode().str.strip()

# Define a mapping
taxa_mapping = {
    'Primate': 'Primates',
}

# Apply mapping
df_taxa_standardized = df_taxa.map(lambda x: taxa_mapping.get(x, x))  # if no mapping, keep original

# Step 3: Now you have clean standardized Taxa Orders
taxa_counts = df_taxa_standardized.value_counts()

# Quick check
print(taxa_counts.head())

In [ ]:
# Set style
sns.set_theme(style="whitegrid")

# Plot
plt.figure(figsize=(12, 7))
sns.barplot(
    y=taxa_counts.index,
    x=taxa_counts.values,
    color="C0"
)

plt.title('Distribution of Taxa Orders Studied', fontsize=18)
plt.xlabel('Number of Papers', fontsize=14)
plt.ylabel('Taxa Order', fontsize=14)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Limit to top 8 orders
top_n = 8
if len(taxa_counts) > top_n:
    top_taxa = taxa_counts[:top_n]
    other = taxa_counts[top_n:].sum()
    taxa_counts = pd.concat([top_taxa, pd.Series({"Other": other})])

# Pie chart
plt.figure(figsize=(8, 8))
plt.pie(
    taxa_counts.values,
    labels=taxa_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    explode=[0.05]*len(taxa_counts),
    colors=sns.color_palette('viridis', len(taxa_counts))
)

plt.title('Top Taxa Orders Studied')
plt.tight_layout()
plt.show()

## Species

In [ ]:
species_counts = df['Species'].value_counts()
print(species_counts.head())

In [ ]:
# Set style
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))
sns.barplot(
    y=species_counts.index,
    x=species_counts.values,
    color="C0"
)

plt.title('Species Studied', fontsize=18)
plt.xlabel('Number of Papers', fontsize=14)
plt.ylabel('Species', fontsize=14)

plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))
plt.pie(
    species_counts.values,
    labels=species_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    explode=[0.05]*len(species_counts),
    colors=sns.color_palette('viridis', len(species_counts))
)

plt.title('Top Species Studied')
plt.tight_layout()
plt.show()

## Region Broad

In [ ]:
region_broad_counts = df['Region Broad'].value_counts()

# Set style
sns.set_theme(style="whitegrid")

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(
    x=region_broad_counts.values,
    y=region_broad_counts.index,
    color="C0"
)

plt.title('Distribution of Region Broad', fontsize=18)
plt.xlabel('Number of Papers', fontsize=14)
plt.ylabel('Region Broad', fontsize=14)

plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart
plt.figure(figsize=(8, 8))
plt.pie(
    region_broad_counts.values,
    labels=region_broad_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    colors=sns.color_palette('viridis', len(region_broad_counts))
)

plt.title('Region Broad Distribution')
plt.tight_layout()
plt.show()

## Region Specific

In [ ]:
df['Region Specific'].unique()

In [ ]:
region_specific = df['Region Specific'].dropna()
region_specific = region_specific.str.split(r'[;,]').explode().str.strip()

# Map regions
region_mapping = {
    'USA': 'United States of America',
    'US': 'United States of America',
    'UK': 'United Kingdom',
    'Malaysian': 'Malaysia',
    'Republic of Korea': 'South Korea',
}
region_specific = region_specific.map(lambda x: region_mapping.get(x, x))
region_specific_counts = region_specific.value_counts()

In [ ]:
# Set style
sns.set_theme(style="whitegrid")

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(
    x=region_specific_counts.values,
    y=region_specific_counts.index,
    color="C0"
)

plt.title('Distribution of Region Specific', fontsize=18)
plt.xlabel('Number of Papers', fontsize=14)
plt.ylabel('Region Specific', fontsize=14)

plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
map_df = pd.DataFrame({'country': region_specific_counts.index, 'count': region_specific_counts.values})

fig = px.scatter_geo(
    map_df,
    locations="country",
    locationmode="country names",
    size="count",
    color="count",
    hover_name="country",
    hover_data={"count": True},
    projection="natural earth",
    color_continuous_scale=[(0, "red"), (1, "green")],  # Red to green scale
    range_color=[1, 4],  # Fix the scale from 1 to 20
    title="Event Count by Country"
)

# Customize the map appearance
fig.update_geos(
    showcoastlines=True,
    coastlinecolor="LightGray",
    showland=True,
    landcolor="WhiteSmoke",
    showocean=True,
    oceancolor="WhiteSmoke",
    showframe=False
)

# Add a color scale bar
fig.update_layout(
    coloraxis_colorbar=dict(
        title="Count",
        tickvals=[1, 10, 20],
        ticktext=["1", "10", "20"],
    )
)

fig.show()

In [ ]:
# Step 1: Mapping Region Specific to Continent
continent_mapping = {
    'China': 'Asia',
    'Japan': 'Asia',
    'Indonesia': 'Asia',
    'Malaysia': 'Asia',
    'South Korea': 'Asia',
    'United Arab Emirates': 'Asia',

    'United States of America': 'North America',

    'Germany': 'Europe',
    'United Kingdom': 'Europe',
    'Belarus': 'Europe',
    'Lithuania': 'Europe',
    'Latvia': 'Europe',

    'Cameroon': 'Africa',
    'Madagascar': 'Africa',
}

region_continents = region_specific_counts.index.map(lambda x: continent_mapping.get(x, 'Other'))

continent_colors = {
    'North America': 'royalblue',
    'South America': 'forestgreen',
    'Europe': 'gold',
    'Asia': 'crimson',
    'Africa': 'darkorange',
    'Oceania': 'mediumvioletred',
    'Other': 'gray'
}

colors = [continent_colors.get(continent, 'gray') for continent in region_continents]

# Plot
plt.figure(figsize=(8, 8))
plt.pie(
    region_specific_counts.values,
    labels=region_specific_counts.index,
    autopct='%1.1f%%',
    counterclock=False,
    explode=[0.05] * len(region_specific_counts),
    colors=colors
)

plt.title('Region Specific Distribution (Colored by Continent)', fontsize=16)
plt.tight_layout()
plt.show()

# Performance Information

## Evaluation Method

In [ ]:
# Split multiple methods and clean
df_evaluation = df['Evaluation Method'].dropna()

In [ ]:
evaluation_counts = df_evaluation.value_counts()

# Plot - Bar Plot
plt.figure(figsize=(12, 7))
sns.barplot(
    x=evaluation_counts.values,
    y=evaluation_counts.index,
    color="C0"
)

plt.title('Distribution of Evaluation Methods', fontsize=18)
plt.xlabel('Number of Papers', fontsize=14)
plt.ylabel('Evaluation Method', fontsize=14)

plt.grid(True, axis='x')
plt.tight_layout()
plt.show()

## Eval Metrics

In [ ]:
# Build the metric distribution from the extracted data.
metric_cols = ["Accuracy", "Precision", "Recall", "Specificity", "F1 Score", "UAR"]
df_melted_metrics = df.melt(value_vars=metric_cols, var_name="Metric", value_name="Score").dropna(subset=["Score"])
plt.figure(figsize=(12, 7))
sns.violinplot(x="Metric", y="Score", data=df_melted_metrics, color="C0", inner="box")

plt.title('Distribution of Evaluation Metrics (Violin)', fontsize=18)
plt.xlabel('Evaluation Metric', fontsize=14)
plt.ylabel('Score', fontsize=14)

plt.grid(True, axis='y')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


In [ ]:
def plot_sankey_layers(df,
                       first_layer_col="Title",
                       middle_layer_col="ML Algorithm Specific",
                       metric_col="Accuracy",
                       metric_format="{:.4f}"):
    """
    General Sankey plot: first_layer → middle_layer → metric_col

    Parameters:
    - df: pandas.DataFrame
        Your dataframe.
    - first_layer_col: str
        Column name for the first layer (default 'Title').
    - middle_layer_col: str
        Column name for the middle layer (default 'ML Algorithm Specific').
    - metric_col: str
        Column name for the final layer (metric) (default 'Accuracy').
    - metric_format: str
        Format string to format metric values (default '{:.4f}').
    """
    required_cols = [first_layer_col, middle_layer_col, metric_col]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in dataframe.")

    df_exploded = df.dropna(subset=[first_layer_col, middle_layer_col, metric_col]).copy()

    df_exploded[middle_layer_col] = df_exploded[middle_layer_col].str.split(r'[,;]')
    df_exploded = df_exploded.explode(middle_layer_col)
    df_exploded[middle_layer_col] = df_exploded[middle_layer_col].str.strip()

    # Build layers
    first_layer = df_exploded[first_layer_col]
    middle_layer = df_exploded[middle_layer_col]
    metric_layer = df_exploded[metric_col].map(lambda x: f"{metric_col}: {metric_format.format(x)}")

    # Create unique nodes
    all_nodes = pd.concat([first_layer, middle_layer, metric_layer]).dropna().unique().tolist()
    node_indices = {node: idx for idx, node in enumerate(all_nodes)}

    # Create links
    source1 = first_layer.map(node_indices).tolist()
    target1 = middle_layer.map(node_indices).tolist()
    source2 = middle_layer.map(node_indices).tolist()
    target2 = metric_layer.map(node_indices).tolist()

    source = source1 + source2
    target = target1 + target2
    value = [1] * len(source)

    # Plot
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=20,
            thickness=15,
            line=dict(color="black", width=0.5),
            label=all_nodes,
        ),
        link=dict(
            source=source,
            target=target,
            value=value
        ))])

    fig.update_layout(title_text=f"{first_layer_col} → {middle_layer_col} → {metric_col}", font_size=10)
    fig.show()


In [ ]:
df.columns

In [ ]:
plot_sankey_layers(df, first_layer_col="Title", middle_layer_col="ML Algorithm Specific", metric_col="Accuracy")

In [ ]:
plot_sankey_layers(df, first_layer_col="Title", middle_layer_col="ML Algorithm Specific", metric_col="Precision")

In [ ]:
plot_sankey_layers(df, first_layer_col="Title", middle_layer_col="ML Algorithm Specific", metric_col="Recall")

In [ ]:
plot_sankey_layers(df, first_layer_col="Title", middle_layer_col="ML Algorithm Specific", metric_col="Specificity")

In [ ]:
plot_sankey_layers(df, first_layer_col="Title", middle_layer_col="ML Algorithm Specific", metric_col="F1 Score")

In [ ]:
plot_sankey_layers(df, first_layer_col="Title", middle_layer_col="ML Algorithm Specific", metric_col="UAR")